# Pose completion: PoseD-Flow's protocol (their Tab. 2), PoseD-Flow and RSGM

The rotations of a body part are hidden, and a model fills them in: 10 hypotheses per pose. Everything follows PoseD-Flow (Nadar, Foti, Birdal, CVPR 2026; their `pose_completion.py`), `hp/evaluation/completion.py`:

- **Data**: their AMASS test set (SMPL fits, DPoser's preprocessing), every 10th pose, 5000 poses (their batches of 500, the last one dropped).
- **Occlusions**: left leg, legs, arms, trunk (their `BodyPartIndices`; "trunk" is the three spine joints and the two shoulders).
- **Metrics** on SMPL-X neutral, betas = 0 (their body model): MPVPE over the vertices of the hidden part, MPJPE over the hidden joints, mean, std and min over the 10 hypotheses; APD of the hidden joints over the hypotheses (cm). Their table reports MPVPE mean ± std and APD. Added here: min over the hypotheses (best of 10), MPJPE, MGEO of the hidden joints (rad).
- **Method**: every model is completed by **D-Flow** (`hp/models/dflow.py`) with their settings: the start of the ODE is optimized so that the visible joints of the result match (geodesic loss), 200 Adam steps (300 for the arms), lr 0.1, 100 Euler steps of the ODE, running cost 1e-3, and the start mixes noise with the sampling ODE run from the observed pose. Only the ODE inside differs: PoseD-Flow's field, or RSGM's probability flow. Their RiemannianAdam is Adam on the tangent coordinates here.

**Joints.** RSGM has 19 joints (no toes): a hidden toe is set to rest. The columns *toes at rest* set the toes to rest in the ground truth and in every completion, which compares 19- and 21-joint models alike.

**The check of the protocol**: PoseD-Flow's checkpoint should give about their PoseRFM row (left leg 83.81 ± 29.07, APD 6.02). Differences come from the optimizer (Adam vs RiemannianAdam) and the random draws.

**Computing.** D-Flow optimizes every hypothesis (about 0.2–0.3 s per step for 500 poses on mps: several hours per model). Run `scripts/evaluate_completion.py` (e.g. on a server); this notebook reads its cache and computes only if `COMPUTE_DEVICE` is set.

In [ ]:
import sys
from pathlib import Path

SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from hp.evaluation.completion import OCCLUSIONS, PAPER, Body, cached_metrics, cached_run, summary, test_poses

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_posedflow_c10"  # as scripts/evaluate_completion.py
CACHE = "~/data/experiments/completion/cache"
MODELS = {  # as scripts/evaluate_completion.py
    "PoseD-Flow: their checkpoint": dict(posedflow="~/data/posedflow_ckpt/checkpoints/epoch-009_step-0_loss-0.0000.ckpt"),
    "RSGM 512x4: their c10": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001"),
    "RSGM 1536x6: their c10": dict(run="~/data/experiments/rsgm_posedflow_s10/run_0001"),
    "RSGM 512x4: their c10, seed 11": dict(run="~/data/experiments/rsgm_posedflow_small_s11/run_0001"),
    "RSGM 512x4: their c10, seed 12": dict(run="~/data/experiments/rsgm_posedflow_small_s12/run_0001"),
    "RSGM 512x4: their full": dict(run="~/data/experiments/full/rsgm_posedflow_small_s10/run_0001"),
    # the same network with other ODEs inside D-Flow (the samplers of the Pareto comparison): half the network calls
    "RSGM 512x4: their c10, ODE Euler 50": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001",
                                                 prior=dict(ode_steps=50)),
    "RSGM 512x4: their c10, Heun EDM 50 NFE": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001",
                                                   prior=dict(sampler="heun", schedule="edm", ode_steps=25)),
    # DPoser (Euclidean diffusion on axis-angle) through the same D-Flow: separates the prior from their own completion
    "DPoser: their full": dict(run="~/data/experiments/full/dposer_posedflow_s10/run_0001"),
    "DPoser: their c10, RSGM budget": dict(run="~/data/experiments/dposer_posedflow_rsgm_budget_s10/run_0001"),
    # their start pushes the observation forward through the ODE; in R^n that leaves the typical set of N(0, I)
    "DPoser: their full, inverted start": dict(run="~/data/experiments/full/dposer_posedflow_s10/run_0001",
                                               prior=dict(invert=True)),
    "RSGM 512x4: their c10, inverted start": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001",
                                                  prior=dict(invert=True)),
    # DPoser's own completion (DPoser-X, the method of the DPoser rows of PoseD-Flow's table), not D-Flow
    "DPoser: their full, DPoser method": dict(run="~/data/experiments/full/dposer_posedflow_s10/run_0001",
                                              method="dposer"),
    "DPoser: their c10, RSGM budget, DPoser method": dict(
        run="~/data/experiments/dposer_posedflow_rsgm_budget_s10/run_0001", method="dposer"),
    # inpainting with the reverse process of RSGM (hp/models/inpainting.py): one point per Optuna study
    # (config/completion_tuning/rsgm_inpaint_*.yaml), chosen on 1000 validation poses by the rule of the studies (the
    # lowest mean among the points of the front with APD >= 6.40, that of D-Flow with Heun EDM 50)
    "RSGM 512x4: their c10, inpaint t52": dict(
        run="~/data/experiments/rsgm_posedflow_small_s10/run_0001", method="inpaint",
        prior=dict(steps=400, schedule="edm", guidance=0.057708678925692564, resample=5, replace=False)),
    "RSGM deep: their c10, inpaint t75": dict(
        run="~/data/experiments/rsgm_posedflow_deep_s10/run_0001", method="inpaint",
        prior=dict(steps=200, schedule="edm", guidance=0.09753705685072556, resample=5, replace=False)),
    "RSGM abl residual: their c10, inpaint t81": dict(
        run="~/data/experiments/rsgm_posedflow_abl_residual_s10/run_0001", method="inpaint",
        prior=dict(steps=400, schedule="edm", guidance=0.04934967328387599, resample=5, replace=False)),
    "RSGM abl norm: their c10, inpaint t118": dict(
        run="~/data/experiments/rsgm_posedflow_abl_norm_s10/run_0001", method="inpaint",
        prior=dict(steps=200, schedule="edm", guidance=0.09515197497012039, resample=5, replace=False)),
}
POSES = None  # None: their 5000 test poses; a number: the first n (a separate cache entry)
COMPUTE_DEVICE = None  # None: only read the cache; "mps" / "cuda:0": compute what is missing
# ----------------------------------------------------------------------------------------

## 1. The completions and their metrics

In [ ]:
gt = test_poses(ROTATIONS, POSES)
body = Body()
results, rest = {}, {}
for name, spec in MODELS.items():
    for occlusion in OCCLUSIONS:
        pred = cached_run(Path(CACHE), name, spec, gt, occlusion, COMPUTE_DEVICE, rotations=ROTATIONS)
        if pred is None:
            print(f"{name} / {occlusion}: not computed yet")
            continue
        plain, toes_at_rest = cached_metrics(Path(CACHE), name, occlusion, pred, gt, body)  # computed once
        results[name, occlusion], rest[name, occlusion] = summary(plain), summary(toes_at_rest)
print(f"{len(gt)} test poses, {len(results)} of {len(MODELS) * len(OCCLUSIONS)} (model, occlusion) done")

## 2. Next to their Tab. 2

Per occlusion: their rows (MPVPE mean ± std, APD), then ours with the same columns and the added ones. The std is over the 10 hypotheses of a pose (the spread of the completions), averaged over the poses, as theirs; it is not an error bar.

In [ ]:
head = f"{'':34s} {'MPVPE':>15s} {'APD':>6s} {'best of 10':>11s} {'MPJPE':>8s} {'MGEO':>7s} {'toes at rest':>13s}"
for occlusion in OCCLUSIONS:
    print(f"\n{occlusion}\n{head}")
    for name, rows in PAPER.items():
        m, s, a = rows[occlusion]
        print(f"{name + ' (paper)':34s} {f'{m:.2f} ± {s:.2f}':>15s} {a:6.2f}")
    for name in MODELS:
        r = results.get((name, occlusion))
        if r is None:
            continue
        t = rest[name, occlusion]
        mpvpe = f"{r['mpvpe_mean']:.2f} ± {r['mpvpe_std']:.2f}"
        print(f"{name:34s} {mpvpe:>15s} {r['apd']:6.2f} {r['mpvpe_min']:11.2f} "
              f"{r['mpjpe_mean']:8.2f} {r['mgeo_mean']:7.3f} {t['mpvpe_mean']:13.2f}")
print("\nMPVPE, best of 10, MPJPE in mm (hidden part); APD in cm; MGEO in rad. Ours: D-Flow with their settings.")

## 3. Reading

- **The check**: PoseD-Flow's checkpoint against their PoseRFM row. A gap of a few mm is the optimizer and the draws; a large gap means the protocol differs from theirs.
- **RSGM against PoseD-Flow** is the same guidance with another prior. RSGM was trained on their c10 (1.86M poses), their checkpoint on all 18.6M.
- **MPVPE assumes one right answer** (their own remark): a plausible completion other than the ground truth counts as an error. Best of 10 and APD show the other side: whether the right pose is among the hypotheses, and how different they are.
- **Toes at rest** is the fair column between 19 and 21 joints; for the trunk and the arms it is the same as MPVPE (the toes are not hidden).

## 4. Pictures of the completions

As Fig. 4 of PoseRFM: the **visible** part of the body in blue, the **hidden** part, filled by a method, in pink; K hypotheses of every method side by side and the ground truth last (`hp/visualization/completion_figure.py`). The pink part is the hidden part of the metric (the vertices of MPVPE), and the number under a hypothesis is its MPVPE.

Nothing is recomputed: the completions are read from the cache of section 1, so every method that `scripts/evaluate_completion.py` (or the SDS scripts) has run on the test set can be shown, also those not in `MODELS`; **refresh** looks again (e.g. after results were copied from a server).

- **methods**: how many are compared (default `N_COMPARE` = 2); **method 1 … N**: each picked from a drop-down that shows, for the chosen occlusion, its averages over the 5000 test poses (MPVPE mean, APD, best of 10: the numbers of section 2), on the same test pose. Every method is a line of its own: its hypotheses, then the ground truth, and on the left its name with the MPVPE (mean of the 10 hypotheses) and the APD (over the 10) of this pose.
- **occlusion**, **pose** (0 … 4999): the case. **pick** chooses a pose for you: *random*; *hard for 1* / *easy for 1*: one of the 50 poses with the highest / lowest mean error of method 1; *1 better than 2* / *2 better than 1*: one of the 50 poses where the two differ most.
- **hypotheses** k of the 10, chosen as *first* (the first k), *diverse* (farthest point sampling on the joints: shows the spread) or *best to worst* (k spread over the 10 sorted by error).
- **views**, **GT ghost** (the ground truth faintly in grey behind every hypothesis), **errors** (MPVPE under every hypothesis, MPVPE and APD of the pose by the name). The text above the picture also gives best of 10.
- **Save** writes the picture (PNG and PDF) to `SAVE_DIR`.

The last cell draws a fixed figure for the paper from `FIGURE` (no panel needed), as lines per method (`layout="rows"`) or as Fig. 4 of PoseRFM (`"columns"`).

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from hp.visualization.completion_figure import (CHOICES, GT, PICKS, cached_labels, completion_figure,
                                                completion_rows, load_errors, method_option, pick_pose,
                                                render_row)

# ---- parameters ----------------------------------------------------------------------
N_COMPARE = 2                                     # methods side by side
DEFAULT_METHODS = ["DPoser: their full, DPoser method", "RSGM 512x4: their c10"]
SHORT_NAMES = {                                   # under the pictures; the others keep their labels
    "PoseD-Flow: their checkpoint": "PoseD-Flow",
    "DPoser: their full, DPoser method": "DPoser",
    "RSGM 512x4: their c10": "RSGM + D-Flow (ours)",
    "RSGM 512x4: their c10, Heun EDM 50 NFE": "RSGM + D-Flow, Heun (ours)",
    "RSGM 1536x6 lik: their full, SDS (validation trial 5)": "RSGM + SDS (ours)",
}
VIEWS = {"front": (0,), "side": (90,), "front and side": (0, 90)}   # azimuths of the renders
SIZE = (300, 420)                                 # pixels of a render (width, height)
SAVE_DIR = "~/data/completion_figures"            # pictures saved from the panel
# ----------------------------------------------------------------------------------------

In [ ]:
L = lambda w: widgets.Layout(width=w)
labels = cached_labels(CACHE)


def defaults(n):
    chosen = [m for m in DEFAULT_METHODS if m in labels] + [m for m in labels if m not in DEFAULT_METHODS]
    return [chosen[i % len(chosen)] for i in range(n)]


count_box = widgets.BoundedIntText(value=N_COMPARE, min=1, max=8, description="methods", layout=L("160px"))
method_boxes = []
methods_area = widgets.VBox()


def options():
    # (shown text, label): every cached method with its averages over the test poses for the chosen occlusion
    return [(method_option(CACHE, m, occlusion_box.value), m) for m in labels]


def set_count(change=None):
    # as many drop-downs as count_box says; those there keep their choice
    n, kept = count_box.value, [box.value for box in method_boxes]
    values = kept[:n] + [v for v in defaults(n) if v not in kept][:max(0, n - len(kept))]
    shown = options()
    method_boxes[:] = [widgets.Dropdown(options=shown, value=v, description=f"method {i + 1}", layout=L("900px"))
                       for i, v in enumerate(values)]
    methods_area.children = method_boxes


def set_options(change=None):
    # the averages of the drop-downs follow the occlusion; the choices stay
    shown = options()
    for box in method_boxes:
        current = box.value
        box.options = shown
        box.value = current if current in labels else labels[0]
occlusion_box = widgets.ToggleButtons(options=list(OCCLUSIONS), value="legs", description="occlusion",
                                      style={"button_width": "90px"})
pose_box = widgets.BoundedIntText(value=0, min=0, max=len(gt) - 1, description="pose", layout=L("180px"))
pick_box = widgets.ToggleButtons(options=PICKS, description="pick", style={"button_width": "120px"})
pick_button = widgets.Button(description="pick pose", icon="random")
k_box = widgets.BoundedIntText(value=4, min=1, max=10, description="hypotheses", layout=L("160px"))
choice_box = widgets.ToggleButtons(options=CHOICES, value="diverse", description="which", style={"button_width": "100px"})
views_box = widgets.ToggleButtons(options=list(VIEWS), description="views", style={"button_width": "110px"})
ghost_box = widgets.Checkbox(value=False, description="GT ghost")
errors_box = widgets.Checkbox(value=True, description="errors")
show_button = widgets.Button(description="Show", button_style="primary", icon="play")
save_button = widgets.Button(description="Save", icon="save")
refresh_button = widgets.Button(description="refresh", icon="refresh", tooltip="look for cached methods again")
out, msg = widgets.Output(), widgets.Output()
last = {}
rng = np.random.default_rng()


def say(text):
    with msg:
        msg.clear_output(wait=True)
        print(text)


def methods():
    return list(dict.fromkeys(box.value for box in method_boxes))   # in order, without repeats


def pick(_):
    errors = [load_errors(CACHE, m, occlusion_box.value) for m in methods()]
    try:
        pose_box.value = pick_pose(pick_box.value, errors, rng)
        show(None)
    except ValueError as error:
        say(str(error))


def show(_):
    occlusion, pose = occlusion_box.value, pose_box.value
    say(f"rendering {occlusion}, pose {pose} ...")
    row = completion_rows(CACHE, methods(), occlusion, pose, k_box.value, body, gt, how=choice_box.value)
    missing = [m["label"] for m in row["methods"] if m["poses"] is None]
    images = render_row(row, body, azimuths=VIEWS[views_box.value], size=SIZE, ghost=ghost_box.value)
    fig = completion_figure([row], [images], names=SHORT_NAMES, errors=errors_box.value, layout="rows")
    last.update(fig=fig, name=f"{occlusion}_pose{pose}_k{k_box.value}_{choice_box.value.replace(' ', '-')}")
    with out:
        out.clear_output(wait=True)
        display(fig)
    plt.close(fig)
    shown = [m for m in row["methods"] if m["mpvpe"] is not None]
    width = max((len(m["label"]) for m in shown), default=0)
    lines = [f"  {m['label']:<{width}}  MPVPE {m['mpvpe']:6.1f} mm  APD {m['apd']:5.2f} cm  best of 10 {m['best']:6.1f} mm"
             for m in shown]
    say(f"{occlusion}, pose {pose} (MPVPE: mean of the 10 hypotheses of this pose):\n" + "\n".join(lines)
        + (f"\nnot computed for this occlusion: {missing}" if missing else ""))


def save(_):
    if "fig" not in last:
        return say("nothing shown yet")
    folder = Path(SAVE_DIR).expanduser()
    folder.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "pdf"):
        last["fig"].savefig(folder / f"{last['name']}.{suffix}", dpi=200, bbox_inches="tight")
    say(f"saved {folder / last['name']}.png / .pdf")


def refresh(_):
    global labels
    labels = cached_labels(CACHE)
    set_options()
    say(f"{len(labels)} cached methods")


set_count()
count_box.observe(set_count, names="value")
occlusion_box.observe(set_options, names="value")
pick_button.on_click(pick)
show_button.on_click(show)
save_button.on_click(save)
refresh_button.on_click(refresh)
display(widgets.VBox([
    count_box, methods_area,
    occlusion_box,
    widgets.HBox([pose_box, pick_button]), pick_box,
    widgets.HBox([k_box, choice_box]),
    widgets.HBox([views_box, ghost_box, errors_box]),
    widgets.HBox([show_button, save_button, refresh_button]),
    msg, out,
]))

In [ ]:
# A fixed figure for the paper: rows (occlusion, test pose), the same methods in every row
FIGURE = dict(
    methods=["DPoser: their full, DPoser method", "RSGM 512x4: their c10"],
    rows=[("legs", 4253), ("arms", 3184)],
    k=4, which="diverse", views="front", ghost=False, errors=False,
    layout="columns",                              # "rows": a line per method; "columns": as Fig. 4 of PoseRFM
    file="completion_legs_arms",                   # in SAVE_DIR, as .png and .pdf
)

rows = [completion_rows(CACHE, FIGURE["methods"], occlusion, pose, FIGURE["k"], body, gt, how=FIGURE["which"])
        for occlusion, pose in FIGURE["rows"]]
images = [render_row(r, body, azimuths=VIEWS[FIGURE["views"]], size=SIZE, ghost=FIGURE["ghost"]) for r in rows]
fig = completion_figure(rows, images, names=SHORT_NAMES, errors=FIGURE["errors"], layout=FIGURE["layout"])
folder = Path(SAVE_DIR).expanduser()
folder.mkdir(parents=True, exist_ok=True)
for suffix in ("png", "pdf"):
    fig.savefig(folder / f"{FIGURE['file']}.{suffix}", dpi=200, bbox_inches="tight")
print(f"saved {folder / FIGURE['file']}.png / .pdf")